<a href="https://colab.research.google.com/github/arquivosdagiullia/CursovigsaudeR/blob/main/Atividade%20dia%2004/04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Atividade do dia 04-04-2026

In [8]:
for (pkg in required_packages) {
  if (!require(pkg, character.only = TRUE)) {
    message(paste("Instalando e carregando:", pkg))
    install.packages(pkg, dependencies = TRUE)
    library(pkg, character.only = TRUE)
  } else {
    message(paste("Pacote já carregado:", pkg))
  }
}

Pacote já carregado: ggplot2

Pacote já carregado: dplyr

Pacote já carregado: tidyr



In [13]:
#' Gerencia instalação e carregamento de pacotes
PackageManager <- R6::R6Class("PackageManager",
  public = list(
    # CREATE - Instalar pacotes
    install_packages = function(packages, dependencies = TRUE) {
      if (length(packages) == 0) {
        message("Nenhum pacote para instalar")
        return(invisible(FALSE))
      }

      for (pkg in packages) {
        if (!self$check_package(pkg)) {
          message(paste("Instalando:", pkg))
          install.packages(pkg, dependencies = dependencies)
          self$load_package(pkg)
        }
      }
      return(invisible(TRUE))
    },

    # READ - Verificar pacotes
    check_package = function(pkg) {
      require(pkg, character.only = TRUE, quietly = TRUE)
    },

    # UPDATE - Atualizar pacotes
    update_packages = function(packages = NULL) {
      if (is.null(packages)) {
        message("Atualizando todos os pacotes...")
        update.packages(ask = FALSE)
      } else {
        for (pkg in packages) {
          if (self$check_package(pkg)) {
            message(paste("Atualizando:", pkg))
            install.packages(pkg, dependencies = TRUE)
          }
        }
      }
    },

    # DELETE - Remover pacotes
    remove_packages = function(packages) {
      for (pkg in packages) {
        if (self$check_package(pkg)) {
          message(paste("Removendo:", pkg))
          remove.packages(pkg)
        } else {
          message(paste("Pacote não encontrado:", pkg))
        }
      }
    },

    # Utilitário - Carregar pacote
    load_package = function(pkg) {
      library(pkg, character.only = TRUE, quietly = TRUE)
      message(paste("Carregado:", pkg))
    },

    # Listar pacotes instalados
    list_installed = function(pattern = NULL) {
      installed <- installed.packages()
      packages <- installed[, "Package"]

      if (!is.null(pattern)) {
        packages <- packages[grepl(pattern, packages, ignore.case = TRUE)]
      }

      return(packages)
    }
  )
)




In [17]:
# modules/package_manager.R

#' Gerencia instalação e carregamento de pacotes
PackageManager <- R6::R6Class("PackageManager",
  public = list(
    # CREATE - Instalar pacotes
    install_packages = function(packages, dependencies = TRUE) {
      if (length(packages) == 0) {
        message("Nenhum pacote para instalar")
        return(invisible(FALSE))
      }

      for (pkg in packages) {
        if (!self$check_package(pkg)) {
          message(paste("Instalando:", pkg))
          install.packages(pkg, dependencies = dependencies)
          self$load_package(pkg)
        }
      }
      return(invisible(TRUE))
    },

    # READ - Verificar pacotes
    check_package = function(pkg) {
      require(pkg, character.only = TRUE, quietly = TRUE)
    },

    # UPDATE - Atualizar pacotes
    update_packages = function(packages = NULL) {
      if (is.null(packages)) {
        message("Atualizando todos os pacotes...")
        update.packages(ask = FALSE)
      } else {
        for (pkg in packages) {
          if (self$check_package(pkg)) {
            message(paste("Atualizando:", pkg))
            install.packages(pkg, dependencies = TRUE)
          }
        }
      }
    },

    # DELETE - Remover pacotes
    remove_packages = function(packages) {
      for (pkg in packages) {
        if (self$check_package(pkg)) {
          message(paste("Removendo:", pkg))
          remove.packages(pkg)
        } else {
          message(paste("Pacote não encontrado:", pkg))
        }
      }
    },

    # Utilitário - Carregar pacote
    load_package = function(pkg) {
      library(pkg, character.only = TRUE, quietly = TRUE)
      message(paste("Carregado:", pkg))
    },

    # Listar pacotes instalados
    list_installed = function(pattern = NULL) {
      installed <- installed.packages()
      packages <- installed[, "Package"]

      if (!is.null(pattern)) {
        packages <- packages[grepl(pattern, packages, ignore.case = TRUE)]
      }

      return(packages)
    }
  )
)
# modules/crud_manager.R

#' Gerencia operações CRUD para qualquer tipo de dado
CRUDManager <- R6::R6Class("CRUDManager",
  public = list(
    data = NULL,
    storage_path = NULL,

    # CREATE - Adicionar registro
    create = function(record, id_field = "id") {
      if (is.null(self$data)) {
        self$data <- data.frame()
      }

      # Gerar ID automático se não existir
      if (is.null(record[[id_field]])) {
        record[[id_field]] <- ifelse(nrow(self$data) == 0, 1, max(self$data[[id_field]]) + 1)
      }

      self$data <- rbind(self$data, record)
      self$save_data()
      message(paste("Registro criado com ID:", record[[id_field]]))
      return(record)
    },

    # READ - Buscar registros
    read = function(id = NULL, filter = NULL) {
      if (is.null(self$data)) {
        return(data.frame())
      }

      result <- self$data

      if (!is.null(id)) {
        result <- result[result$id == id, ]
      }

      if (!is.null(filter)) {
        for (col in names(filter)) {
          if (col %in% names(result)) {
            result <- result[result[[col]] == filter[[col]], ]
          }
        }
      }

      return(result)
    },

    # UPDATE - Atualizar registro
    update = function(id, updates) {
      if (is.null(self$data) || !any(self$data$id == id)) {
        message(paste("ID não encontrado:", id))
        return(invisible(FALSE))
      }

      for (col in names(updates)) {
        if (col %in% names(self$data)) {
          self$data[self$data$id == id, col] <- updates[[col]]
        }
      }

      self$save_data()
      message(paste("Registro", id, "atualizado com sucesso"))
      return(invisible(TRUE))
    },

    # DELETE - Remover registro
    delete = function(id) {
      if (is.null(self$data) || !any(self$data$id == id)) {
        message(paste("ID não encontrado:", id))
        return(invisible(FALSE))
      }

      self$data <- self$data[self$data$id != id, ]
      self$save_data()
      message(paste("Registro", id, "removido com sucesso"))
      return(invisible(TRUE))
    },

    # Persistência de dados
    save_data = function() {
      if (!is.null(self$storage_path)) {
        saveRDS(self$data, self$storage_path)
      }
    },

    load_data = function() {
      if (!is.null(self$storage_path) && file.exists(self$storage_path)) {
        self$data <- readRDS(self$storage_path)
      }
    },

    initialize = function(storage_path = NULL) {
      self$storage_path <- storage_path
      self$load_data()
    }
  )
)